# TP 3 : Fine-tuning avec LoRA (Low-Rank Adaptation)

## Objectifs du TP

Ce TP vous permettra de :
- Comprendre le principe du fine-tuning adaptatif avec LoRA
- Mettre en pratique le fine-tuning d'un modèle de langage avec quantization 4-bit
- Adapter un modèle pré-entraîné pour une tâche spécifique (support client bancaire)
- Tester et évaluer le modèle fine-tuné

## Prérequis

- Notions de base sur les modèles de langage
- Familiarité avec Python et PyTorch
- Compréhension du format de données JSON/JSONL

---


## Section 1 : Installation des dépendances

Dans cette section, nous installons les bibliothèques nécessaires pour le fine-tuning avec LoRA :
- `bitsandbytes` : pour la quantization 4-bit
- `transformers` : bibliothèque Hugging Face pour les modèles
- `accelerate` : pour l'accélération et la distribution
- `peft` : Parametric Efficient Fine-Tuning (inclut LoRA)
- `datasets` : pour charger et manipuler les jeux de données


In [1]:
# Installation des bibliothèques nécessaires
!pip install -U -q bitsandbytes
!pip install -U -q transformers accelerate peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.9 MB/s eta 0:00:00


## Section 2 : Import des bibliothèques

Nous importons les modules nécessaires pour :
- PyTorch : framework de deep learning
- Transformers : chargement des modèles et tokenizers
- Datasets : gestion des données d'entraînement
- PEFT : configuration et utilisation de LoRA


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import DataCollatorForLanguageModeling

## Section 3 : Chargement du modèle et du tokenizer

### 3.1 Choix du modèle de base

Nous utilisons le modèle **CroissantLLMChat-v0.1**, un modèle de langage français pré-entraîné adapté pour le dialogue.

### 3.2 Chargement du tokenizer

Le tokenizer convertit le texte en tokens que le modèle peut comprendre.


In [3]:
# Nom du modèle que nous allons fine-tuner
model_name = "croissantllm/CroissantLLMChat-v0.1"

# Chargement du tokenizer
# Le tokenizer est nécessaire pour convertir le texte en tokens
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"✅ Tokenizer chargé : {model_name}")
print(f"Vocabulaire : {len(tokenizer)} tokens")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


✅ Tokenizer chargé : croissantllm/CroissantLLMChat-v0.1
Vocabulaire : 32002 tokens


### 3.3 Chargement du modèle avec quantization 4-bit

**Pourquoi la quantization 4-bit ?**
- Réduit considérablement la mémoire GPU nécessaire (de ~16GB à ~4GB pour un modèle de 7B)
- Permet d'entraîner sur des GPUs avec moins de mémoire
- Impact minimal sur les performances avec LoRA

**Paramètres importants :**
- `device_map="auto"` : répartit automatiquement le modèle sur les GPUs disponibles
- `load_in_4bit=True` : active la quantization 4-bit via bitsandbytes
- `torch_dtype=torch.float16` : utilise la précision half pour économiser la mémoire


In [4]:
# Chargement du modèle avec quantization 4-bit
# Cette étape peut prendre quelques minutes lors du premier téléchargement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",            # Répartit automatiquement sur GPU/CPU
    load_in_4bit=True,            # Quantization 4-bit (réduit la mémoire de ~75%)
    torch_dtype=torch.float16,    # Précision mixte (half precision)
    trust_remote_code=True        # Nécessaire pour certains modèles personnalisés
)

print("✅ Modèle chargé avec quantization 4-bit")
print(f"Emplacement du modèle : {model.hf_device_map if hasattr(model, 'hf_device_map') else 'GPU'}")


config.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/397M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Modèle chargé avec quantization 4-bit
Emplacement du modèle : {'': 0}


### 3.4 Test du modèle avant finetuning

In [5]:
generation_args = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.3,
    "top_p": 0.90,
    "top_k": 40,
    "repetition_penalty": 1.05,
    "eos_token_id": [tokenizer.eos_token_id, 32000],
}

chat = [
   {"role": "user", "content": "Bonjour, je souhaite ouvrir un compte courant."},
]

chat_input = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(chat_input, return_tensors="pt").to(model.device)
tokens = model.generate(**inputs, **generation_args)

print(tokenizer.decode(tokens[0]))
# print tokens individually
print([(tokenizer.decode([tok]), tok) for tok in tokens[0].tolist()])


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


<s><|im_start|> user
Bonjour, je souhaite ouvrir un compte courant.<|im_end|> 
<|im_start|> assistant
 Bonjour,

Je suis ravi de vous aider à créer votre nouveau compte bancaire en suivant les étapes suivantes :
1. Connectez-vous à votre banque en ligne ou créez un nouveau compte si vous n'en avez pas encore.
2. Rendez-vous sur le site web de la banque, où il y a une section "Compte" qui contient des informations et des instructions pour accéder au processus d'ouverture d'un compte courant.
3. Sur le site, recherchez l'option "Ouvrir un Compte".
4. Suivez les invites pour fournir vos informations personnelles telles que votre nom, adresse e-mail, numéro de téléphone et date de naissance.
5. Vérifiez votre identité en répondant aux questions posées par l'assistant virtuel du système. Vous devrez répondre à quelques autres questions, mais c'est tout.
6. Une fois terminé, cliquez sur "Suivant" pour passer à l'étape suivante.

Une fois connecté, suivez ces étapes supplémentaires :
Remarque

## Section 4 : Chargement et préparation des données

### 4.1 Chargement du dataset

Nous chargeons un dataset de dialogues de support client bancaire au format JSONL.

**Format attendu :** Chaque ligne contient un dialogue au format JSON avec une clé `"chat"` contenant une liste de messages avec `"role"` et `"content"`.


In [6]:
# Chargement des datasets d'entraînement et de validation
# Remplacez ces chemins par les vôtres si nécessaire
dataset = load_dataset("json", data_files={
    "train": "/content/sample_data/bank_customer_support_train.jsonl",
    "validation": "/content/sample_data/bank_customer_support_val.jsonl"
})

print(f"✅ Datasets chargés")
print(f"Nombre d'exemples d'entraînement : {len(dataset['train'])}")
print(f"Nombre d'exemples de validation : {len(dataset['validation'])}")


FileNotFoundError: Unable to find '/content/sample_data/bank_customer_support_train.jsonl'

### 4.2 Préprocessing des dialogues

**Objectif :** Convertir les dialogues en format tokenisé que le modèle peut utiliser.

**Étapes :**
1. Utiliser `apply_chat_template` pour formater le dialogue selon le format attendu par le modèle
2. Tokeniser le texte résultant
3. Tronquer à 512 tokens (longueur maximale)


In [ ]:
def preprocess(example):
    """
    Prétraite un exemple de dialogue pour l'entraînement.

    Args:
        example: Un dictionnaire contenant au minimum une clé "chat"
                 avec une liste de messages {"role": ..., "content": ...}

    Returns:
        Un dictionnaire avec les tokens tokenisés
    """
    # Extraire la liste de messages du dialogue
    chat = example["chat"]  # Format : [{"role": "user", "content": "..."}, ...]

    # Appliquer le template de chat du modèle (ajoute les balises spéciales)
    # add_generation_prompt=True : ajoute un prompt pour la génération de réponse
    chat_input = tokenizer.apply_chat_template(
        chat,
        tokenize=False,  # On veut juste le texte formaté, pas encore tokenisé
        add_generation_prompt=True
    )

    # Tokeniser le texte formaté avec troncature à 512 tokens
    tokens = tokenizer(
        chat_input,
        truncation=True,
        max_length=512
    )

    return tokens

# Appliquer le preprocessing à tous les exemples
tokenized_datasets = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names  # Supprime les colonnes originales
)

print("✅ Datasets préprocessés et tokenisés")
print(f"Exemple de format : {list(tokenized_datasets['train'][0].keys())}")


## Section 5 : Configuration de LoRA

### 5.1 Qu'est-ce que LoRA ?

**LoRA (Low-Rank Adaptation)** est une technique de fine-tuning adaptatif qui :
- Ajoute des matrices de faible rang au lieu de modifier tous les paramètres
- Réduit drastiquement le nombre de paramètres à entraîner (de millions à quelques milliers)
- Permet un fine-tuning efficace en mémoire et en temps
- Les adaptateurs LoRA peuvent être facilement chargés/déchargés

### 5.2 Paramètres de configuration

- `r` (rank) : rang des matrices de décomposition (8-64 typiquement, plus bas = moins de paramètres)
- `lora_alpha` : facteur de scaling (généralement r * 2 à r * 4)
- `lora_dropout` : taux de dropout pour éviter le surapprentissage
- `target_modules` : quels modules du modèle adapter (attention queries/values pour LLaMA)


In [ ]:
# Configuration LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,          # Type de tâche : Language Modeling
    r=8,                                    # Rank : dimension des matrices de décomposition
    lora_alpha=32,                          # Facteur de scaling (typiquement r * 4)
    lora_dropout=0.1,                       # Dropout pour la régularisation
    target_modules=["q_proj", "v_proj"]     # Modules à adapter (attention queries et values)
    # Note : Pour d'autres architectures, ajustez target_modules
    # Exemple LLaMA : ["q_proj", "v_proj", "k_proj", "o_proj"]
)

# Appliquer LoRA au modèle
model = get_peft_model(model, lora_config)

# Afficher le nombre de paramètres entraînables
print("📊 Statistiques des paramètres :")
model.print_trainable_parameters()

# Interprétation :
# - Total params : tous les paramètres du modèle (base + LoRA)
# - Trainable params : seulement les paramètres LoRA (très peu !)
# - Trainable% : pourcentage de paramètres entraînables (<1% typiquement)


### 5.3 Data Collator

Le Data Collator est responsable de créer des batches à partir des exemples tokenisés. Pour le Language Modeling, il :
- Pad les séquences à la même longueur dans un batch
- Crée les labels pour la prédiction du token suivant


In [ ]:
# Data collator pour le Language Modeling
# mlm=False : nous faisons du causal LM (prédiction du prochain token), pas du masked LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal Language Modeling (pas de masking)
)

print("✅ Data collator configuré")


## Section 6 : Configuration de l'entraînement

### 6.1 Training Arguments

Configuration des hyperparamètres et des options d'entraînement.

**Paramètres importants :**
- `per_device_train_batch_size` : taille du batch par GPU (adapte selon votre mémoire GPU)
- `gradient_accumulation_steps` : simule des batches plus grands en accumulant les gradients
- `learning_rate` : taux d'apprentissage (3e-4 est un bon point de départ pour LoRA)
- `num_train_epochs` : nombre d'époques
- `fp16` : utilise la précision mixte pour accélérer l'entraînement


In [ ]:
# Configuration des arguments d'entraînement
training_args = TrainingArguments(
    # Répertoire de sortie pour sauvegarder les checkpoints
    output_dir="./finetuned_croissant_bank_lora_4bit",

    # Taille des batches
    per_device_train_batch_size=2,      # Batch size pour l'entraînement
    per_device_eval_batch_size=2,       # Batch size pour l'évaluation

    # Hyperparamètres d'entraînement
    num_train_epochs=3,                 # Nombre d'époques
    learning_rate=3e-4,                 # Taux d'apprentissage (bon pour LoRA)

    # Options de sauvegarde et logging
    logging_steps=50,                   # Log toutes les 50 étapes
    save_steps=500,                     # Sauvegarder un checkpoint toutes les 500 étapes
    eval_steps=500,                     # Évaluer toutes les 500 étapes
    save_total_limit=2,                 # Garder seulement les 2 derniers checkpoints

    # Optimisations
    fp16=True,                          # Utiliser la précision mixte (accélère l'entraînement)
    gradient_accumulation_steps=4,      # Simule un batch_size de 2*4=8

    # Désactiver les rapports externes (W&B, TensorBoard, etc.)
    report_to="none",

    # Note : décommentez la ligne suivante pour activer l'évaluation périodique
    # evaluation_strategy="steps",
)

print("✅ Arguments d'entraînement configurés")

### 6.2 Initialisation du Trainer

Le `Trainer` de Hugging Face encapsule toute la logique d'entraînement :
- Forward pass, backward pass, optimisation
- Gestion des checkpoints
- Logging et évaluation


In [ ]:
# Initialisation du Trainer
trainer = Trainer(
    model=model,                                    # Modèle avec LoRA
    args=training_args,                             # Arguments d'entraînement
    train_dataset=tokenized_datasets["train"],      # Dataset d'entraînement
    eval_dataset=tokenized_datasets["validation"],  # Dataset de validation
    tokenizer=tokenizer,                            # Tokenizer (pour le padding)
    data_collator=data_collator                     # Collator pour créer les batches
)

print("✅ Trainer initialisé et prêt pour l'entraînement")


## Section 7 : Entraînement du modèle

⚠️ **Attention :** Cette étape peut prendre du temps selon la taille de votre dataset et votre matériel.

Pendant l'entraînement, surveillez :
- La loss d'entraînement (doit diminuer)
- La loss de validation (doit suivre la tendance de l'entraînement, pas diverger)
- Les checkpoints sont sauvegardés automatiquement


In [ ]:
# Lancer l'entraînement
print("🚀 Démarrage de l'entraînement...")
trainer.train()

print("✅ Entraînement terminé !")


## Section 8 : Sauvegarde du modèle fine-tuné

Une fois l'entraînement terminé, nous sauvegardons :
- Les poids LoRA (adaptateurs) : très légers (~quelques MB)
- Le tokenizer : nécessaire pour utiliser le modèle

⚠️ **Note importante :** Seuls les adaptateurs LoRA sont sauvegardés, pas le modèle de base.
Pour utiliser le modèle, vous devrez charger le modèle de base + les adaptateurs LoRA.


In [ ]:
# Sauvegarder les adaptateurs LoRA et le tokenizer
save_path = "./finetuned_croissant_bank_lora_4bit"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Modèle LoRA sauvegardé dans : {save_path}")
print("📦 Les adaptateurs LoRA sont très légers (~quelques MB seulement)")


---

## Section 9 : Test et évaluation du modèle fine-tuné

Dans cette section, nous allons charger le modèle fine-tuné et tester sa capacité à répondre à des questions de support client bancaire.

### 9.1 Imports nécessaires pour le chargement


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel


### 9.2 Configuration des chemins

Pour charger le modèle fine-tuné, nous avons besoin de :
- Le modèle de base (celui que nous avons utilisé pour l'entraînement)
- Le chemin vers les adaptateurs LoRA sauvegardés


In [ ]:
# Configuration des chemins
base_model = "croissantllm/CroissantLLMChat-v0.1"  # Modèle de base
lora_path = "./finetuned_croissant_bank_lora_4bit"  # Chemin vers les adaptateurs LoRA

print(f"Modèle de base : {base_model}")
print(f"Chemin LoRA : {lora_path}")


### 9.3 Chargement du tokenizer

Le tokenizer est identique à celui utilisé pendant l'entraînement.


In [ ]:
# Charger le tokenizer depuis le modèle de base
tokenizer = AutoTokenizer.from_pretrained(base_model)
print("✅ Tokenizer chargé")


### 9.4 Configuration de la quantization 4-bit

Pour l'inférence, nous utilisons la même configuration de quantization que pendant l'entraînement.

**Paramètres de BitsAndBytesConfig :**
- `load_in_4bit=True` : active la quantization 4-bit
- `bnb_4bit_compute_dtype` : type de données pour les calculs
- `bnb_4bit_quant_type="nf4"` : type de quantization (NF4 est optimisé)
- `bnb_4bit_use_double_quant=True` : double quantization pour plus de précision


In [ ]:
# Configuration de la quantization 4-bit pour l'inférence
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # Activer la quantization 4-bit
    bnb_4bit_compute_dtype=torch.float16,     # Type pour les calculs (float16)
    bnb_4bit_quant_type="nf4",                # Type de quantization (NF4 optimisé)
    bnb_4bit_use_double_quant=True,           # Double quantization pour plus de précision
)

print("✅ Configuration de quantization définie")


### 9.5 Chargement du modèle de base

Nous chargeons d'abord le modèle de base avec quantization.


In [ ]:
# Charger le modèle de base avec quantization 4-bit
print("📥 Chargement du modèle de base...")
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",                    # Répartir automatiquement sur GPU
    quantization_config=bnb_config,       # Appliquer la quantization
    trust_remote_code=True                # Nécessaire pour certains modèles
)

print("✅ Modèle de base chargé")


### 9.6 Chargement des adaptateurs LoRA

Une fois le modèle de base chargé, nous y attachons les adaptateurs LoRA que nous avons entraînés.

**Avantage :** Les adaptateurs sont très légers et peuvent être facilement échangés pour différentes tâches.


In [ ]:
# Charger les adaptateurs LoRA sur le modèle de base
print("📥 Chargement des adaptateurs LoRA...")
model = PeftModel.from_pretrained(model, lora_path)

# Mettre le modèle en mode évaluation (désactive dropout, etc.)
model.eval()

print("✅ Modèle fine-tuné chargé et prêt pour l'inférence")


### 9.7 Préparation du prompt de test

Nous préparons un exemple de question de support client pour tester le modèle.

**Format du prompt :** Utilisation du format de chat avec `apply_chat_template` pour respecter le format attendu par le modèle.


In [ ]:
# Préparer un exemple de dialogue de support client
chat = [
    {"role": "user", "content": "Bonjour, je souhaite ouvrir un compte courant."}
]

# Formater le dialogue selon le template du modèle
prompt = tokenizer.apply_chat_template(
    chat,
    tokenize=False,              # Retourner le texte, pas les tokens
    add_generation_prompt=True   # Ajouter le prompt pour la génération
)

print("📝 Prompt formaté :")
print(prompt)
print("\n" + "="*80 + "\n")

# Tokeniser le prompt
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(f"✅ Prompt tokenisé (longueur : {inputs['input_ids'].shape[1]} tokens)")


### 9.8 Génération de la réponse

**Paramètres de génération importants :**
- `max_new_tokens` : nombre maximum de tokens à générer
- `temperature` : contrôle la créativité (bas = plus déterministe, haut = plus créatif)
- `top_p` : sampling par nucléus (filtre les tokens les moins probables)
- `do_sample=True` : active le sampling stochastique
- `repetition_penalty` : pénalise les répétitions (1.0 = pas de pénalité)

⚠️ **Note :** Utilisons `torch.no_grad()` pour désactiver le calcul des gradients (économise la mémoire).


In [ ]:
# Générer la réponse
print("🤖 Génération de la réponse...")
print("="*80)

with torch.no_grad():  # Désactiver les gradients (mode inférence uniquement)
    output = model.generate(
        **inputs,                          # Tokens du prompt
        max_new_tokens=200,                # Maximum 200 nouveaux tokens
        temperature=0.3,                   # Basse température = réponse plus déterministe
        top_p=0.9,                         # Sampling par nucléus (top 90% des probabilités)
        do_sample=True,                    # Activer le sampling stochastique
        repetition_penalty=1.05,           # Légère pénalité contre les répétitions
        eos_token_id=tokenizer.eos_token_id  # Token de fin de séquence
    )

# Décoder et afficher la réponse complète
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)
print("\n" + "="*80)
print("✅ Génération terminée")


---

## Conclusion et prochaines étapes

### Ce que nous avons accompli

✅ Fine-tuning d'un modèle de langage avec LoRA et quantization 4-bit  
✅ Adaptation du modèle pour une tâche spécifique (support client bancaire)  
✅ Test et évaluation du modèle fine-tuné  

### Points clés à retenir

1. **LoRA** : Technique efficace qui permet de fine-tuner avec très peu de paramètres
2. **Quantization 4-bit** : Permet d'utiliser des modèles volumineux sur des GPUs avec moins de mémoire
3. **Format de données** : Important d'utiliser le bon format (chat template) pour les modèles de dialogue

### Pour aller plus loin

- Expérimenter avec différents paramètres LoRA (r, alpha, dropout)
- Tester avec d'autres datasets
- Comparer les performances avec/sans quantization
- Évaluer le modèle sur un dataset de test complet

---

**Fin du TP 3** 🎉
